In [1]:
!pip install ipython-sql
!pip install ipython-sql prettytable

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 453.5 kB/s eta 0:00:04
   --------- ------------------------------ 0.5/2.1 MB 453.5 kB/s eta 0:00:04
   -------------- ------------------------- 0.8/2.1 MB 508.5 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.1 MB 508.5 kB/s eta 0:00:03
   ------------------- -------------------- 1.0/2.1 MB 565.4 kB/s eta 0:00:02
   ------------------------ --------------- 1.3/2.1 MB 615.7 kB/s eta 0:00:02
   ----------------------------- ---------- 1.6/2

In [2]:
%load_ext sql

In [3]:
import csv, sqlite3 
import prettytable 
prettytable.DEFAULT='DEFAULT'
con=sqlite3.connect("my_data1.db")
cur=con.cursor()

In [4]:
%sql sqlite:///my_data1.db

In [5]:
import pandas as pd 
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL",con,if_exists='replace',index=False,method="multi")

101

In [6]:
#DROP THE TABLE IF EXISTST 
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null 

 * sqlite:///my_data1.db
Done.


[]

In [34]:
%sql SELECT DISTINCT(Launch_Site) FROM SPACEXTABLE

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


In [35]:
%sql SELECT * FROM SPACEXTABLE WHERE "launch_site" like 'CCA%' LIMIT 5 

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [18]:
%sql SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)'

 * sqlite:///my_data1.db
Done.


total_payload
45596


In [19]:
%sql SELECT AVG(PAYLOAD_MASS__KG_) AS Average_payload_mass FROM SPACEXTABLE WHERE Booster_Version LIKE 'F9 V1.1%'

 * sqlite:///my_data1.db
Done.


Average_payload_mass
2534.6666666666665


In [22]:
%sql SELECT MIN(DATE) FROM SPACEXTABLE WHERE Landing_Outcome='Success (ground pad)'

 * sqlite:///my_data1.db
Done.


MIN(DATE)
2015-12-22


In [23]:
%sql SELECT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


In [ ]:
%sql SELECT Mission_Outcome , COUNT(*) FROM SPACEXTABLE WHERE Mission_Outcome LIKE 'Success%' OR Mission_Outcome LIKE 'Failure%' GROUP BY Mission_Outcome

 * sqlite:///my_data1.db
Done.


Mission_Outcome,COUNT(*)
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


In [30]:
%sql SELECT Mission_Outcome FROM SPACEXTABLE

 * sqlite:///my_data1.db
Done.


Mission_Outcome
Success
Success
Success
Success
Success
Success
Success
Success
Success
Success


In [31]:
%sql SELECT Booster_Version FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


In [32]:
%sql SELECT substr(Date,6,2) AS Month , Landing_Outcome,Booster_Version , Launch_Site FROM SPACEXTABLE WHERE substr(Date,0,5)='2015' and Landing_Outcome= 'Failure (drone ship)'

 * sqlite:///my_data1.db
Done.


Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [33]:
%sql SELECT Landing_Outcome,Count(*) AS Count_Outcome FROM SPACEXTABLE WHERE Date Between '2010-06-04' and '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count_Outcome DESC 

 * sqlite:///my_data1.db
Done.


Landing_Outcome,Count_Outcome
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
